In [1]:
import os
import sys
from os import path
sys.path.insert(0, "/home/thomasb")
import json
import h5py
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone
#import sat_utils as su
import importlib
#from albatros_analysis.src.utils import orbcomm_utils as outils
from albatros_analysis.src.utils import baseband_utils as butils
#import helper_discrepancies as hd
#importlib.reload(su)
importlib.reload(butils)
#importlib.reload(hd)

<module 'albatros_analysis.src.utils.baseband_utils' from '/home/thomasb/albatros_analysis/src/utils/baseband_utils.py'>

In [23]:
#makes reading file easier
map_blines = {0:'MARS 1-2', 1:'MARS 1-4', 2:'MARS 1-5', 3:'MARS 1-6', 4:'MARS 1-7', 5:'MARS 1-8'}

#start time of batch
batch_start_ts = 1753200150

#path to the timing solution file
path_taus = f'/scratch/thomasb/batch_{batch_start_ts}/fine_timing/timing_solution_15may.h5'

#some parameters
nvis = 120
acclen = 1024
osamp = 64
T_SPECTRA = 4096/250e6

#the conversions from spectrum number into UTC time
UTC_per_spec = 1.638402806904218e-05
UTC_offset = 1753200128.4140258

# my flow is indexed by the name of the data file
# for now here's all the russian file names
fnames_russian = [
    'data_raw_osamp=64_start=1753204614_end=1753204909_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753210629_end=1753210874_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753215755_end=1753216140_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753216629_end=1753216874_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753221813_end=1753222206_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753222730_end=1753223066_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753240890_end=1753241135_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753246576_end=1753246773_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753252612_end=1753252907_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753264668_end=1753264963_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753270944_end=1753271238_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753276664_end=1753277253_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753277632_end=1753277829_chans=1836:1839.npy']


taus = {}
stds, errs = {}, {}
utc = []
for bline in map_blines.values():
    taus[bline] = []
    stds[bline] = []
    errs[bline] = []

#open the file up
with h5py.File(path_taus, 'r') as f:
    #iterate over all the russian filenames
    for name in fnames_russian:
        #get baseband spectrum where data starts
        start_spectrum = f[name]['taus'].attrs['starting_specnum']
        # get whole list of baseband spectra for pulse
        spectra = np.arange(start_spectrum, start_spectrum + nvis*acclen*osamp, acclen*osamp)
        #turn into UTC times and append to list
        utc.append(spectra*UTC_per_spec + UTC_offset)

        #need to extract from the h5 as array
        taus_all_file = f[name]['taus'][:]
        errs_all_file = f[name]['errs'][()]
        for i, bline in map_blines.items():
            #add them to the list, indexed by the baseline
            taus[bline].append(taus_all_file[i,:])
            errs[bline].append(errs_all_file[i])             #the error on the fit  (certainty of value)
            stds[bline].append(np.std(taus_all_file[i,:]))   #std across pulse (variation of tau)

#so now if you want to read a certain pulse on a certain baseline:
pnum = 0
bline = 'MARS 1-2'
time_solution = taus[bline][pnum]
print(time_solution.shape)
utcs = utc[pnum]
print(utcs.shape)

(120,)
(120,)
